<a href="https://colab.research.google.com/github/mideade247/ProjectITC/blob/main/A4RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document YOLO + RAG
## Library Installation

In [30]:
#Install dependencies
!pip install -U ultralytics opencv-python pdf2image img2pdf
!apt-get update -y && apt-get install -y poppler-utils tesseract-ocr
!pip install -U openai tiktoken chromadb pytesseract pypdf
!pip install -U pillow

!pip install -U langchain-openai

!pip install "langchain==0.2.14" "langchain-core==0.2.35" "langchain-community==0.2.12" "langchain-openai==0.1.20" "langchain-text-splitters==0.2.2" pinecone

# !pip install --upgrade --force-reinstall numpy==1.26.4
# !pip install --upgrade --force-reinstall torch torchvision torchaudio
# !pip install --upgrade ultralytics opencv-python


from IPython.display import clear_output
clear_output()
print(" Dependencies installed successfully.")


 Dependencies installed successfully.


## Paths Definition

In [2]:
# Configuration (Local paths)
import os
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"
RAG_STORE = BASE_DIR / "rag_store"

for d in [DATA_DIR, MODELS_DIR, OUTPUT_DIR, RAG_STORE]:
    d.mkdir(parents=True, exist_ok=True)

YAML_NAME = MODELS_DIR / "documents.yaml"
WEIGHTS_NAME = MODELS_DIR / "documents.pt"
FILE_PATH = DATA_DIR

print("Base directory:", BASE_DIR)
print("YAML:", YAML_NAME)
print("Weights:", WEIGHTS_NAME)
print("Data path:", FILE_PATH)


Base directory: /content
YAML: /content/models/documents.yaml
Weights: /content/models/documents.pt
Data path: /content/data


In [2]:
# with open(YAML_NAME) as f: print(f.read())


## YOLO Setup and Training


In [3]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

if YAML_NAME.exists():
    model.train(data=str(YAML_NAME), epochs=50, imgsz=640, project=str(OUTPUT_DIR), name='document_training', cache= False)
    model.save(str(WEIGHTS_NAME))
    print("YOLO model trained and saved.")
else:
    print("YAML file not found at", YAML_NAME)


Ultralytics 8.3.218 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/models/documents.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=document_training2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# FILE_PATH = "/content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/val"

# results = model.predict(source=FILE_PATH, save=True, project=str(OUTPUT_DIR))


## Documents Detection

In [4]:
# Document Detection
# if FILE_PATH.exists():
#     results = model.predict(source=str(FILE_PATH), save=True, project=str(OUTPUT_DIR))
#     print(" Detection complete. Results saved in:", OUTPUT_DIR)
# else:
#     print(" Data directory not found:", FILE_PATH)


# import os

# if os.path.exists(FILE_PATH):
#     results = model.predict(source=FILE_PATH, save=True, project=str(OUTPUT_DIR))
#     print("✅ Detection complete. Results saved in:", OUTPUT_DIR)
# else:
#     print("⚠️ File or folder not found:", FILE_PATH)


from pathlib import Path

FILE_PATH = Path("/content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/train")

if FILE_PATH.exists():
    results = model.predict(source=str(FILE_PATH), save=True, project=str(OUTPUT_DIR))
    print("Detection complete. Results saved in:", OUTPUT_DIR)
else:
    print("File or folder not found:", FILE_PATH)




image 1/141 /content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/train/00d31748-1760379246310-ef56ecee-8780-4127-9931-6576483ffdb5_3.png: 640x480 3 captions, 3 images, 9 paragraphs, 2 titles, 54.3ms
image 2/141 /content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/train/0486834f-1760380567469-025ba741-26eb-4fb7-bd39-df29e21dd837_21.png: 640x512 1 image, 5 paragraphs, 1 title, 108.8ms
image 3/141 /content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/train/05af47a4-1760385601595-1c5593d3-273a-4393-9273-6fddf0bfdb1a_6.png: 640x512 2 captions, 3 images, 7 paragraphs, 2 titles, 8.7ms
image 4/141 /content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/train/074760df-1760380567469-025ba741-26eb-4fb7-bd39-df29e21dd837_29.png: 640x512 8 images, 14 paragraphs, 2 titles, 8.1ms
image 5/141 /content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/train/08029068-1760380567469-025ba741-26eb-4fb7

### Converting Img to Text

In [6]:
# Text Extraction
from PIL import Image
from pdf2image import convert_from_path
from pypdf import PdfReader
import pytesseract

def extract_text_from_pdf(path):
    text_pages = []
    try:
        reader = PdfReader(path)
        for page in reader.pages:
            txt = page.extract_text() or ''
            if not txt.strip():
                images = convert_from_path(path)
                for img in images:
                    txt += pytesseract.image_to_string(img)
            text_pages.append(txt)
    except Exception:
        images = convert_from_path(path)
        for img in images:
            text_pages.append(pytesseract.image_to_string(img))
    return "\n".join(text_pages)

def extract_text_from_image(path):
    img = Image.open(path).convert('RGB')
    return pytesseract.image_to_string(img)

def extract_text(path):
    path = str(path)
    lower = path.lower()
    if lower.endswith('.pdf'):
        return extract_text_from_pdf(path)
    elif lower.endswith(('.png','.jpg','.jpeg','.tif','.tiff','.bmp')):
        return extract_text_from_image(path)
    elif lower.endswith('.txt'):
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                return f.read()
        except Exception:
            return ''
    return ''
print("Extraction utilities loaded successfully.")


Extraction utilities loaded successfully.


#### From the text, lets do some findings

##### Pick a sample text

In [7]:
img_path = "/content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/train/a0fb28d6-1760378000404-38a681d7-57f1-40cb-acc3-0c0e2b09c28b_18.png"


#### Extract the Text

In [8]:
extracted_text = extract_text(img_path)


#### Print A Readable Preview

In [9]:
print(extracted_text)


xviii - PREFACE

Well, how did you do? Once you master Small Talk,

you are guaranteed to:

* Build business

* Make friends

+ Improve networking skills

* Get dates

+ Land jobs

All right—that’s enough small talking. Let’s get

down to business!



### Final Texts extraction and Saving

In [10]:
import os

# Folder containing your PDFs and images
folder = "/content/drive/MyDrive/Dataset/Document_Detection/Document Detection/images/val"

# Output file to save all extracted text
output_file = "/content/extracted_texts.txt"

# Create/open the output file
with open(output_file, "w", encoding="utf-8") as out:
    for f in os.listdir(folder):
        if f.lower().endswith(('.pdf', '.png', '.jpg', '.jpeg')):
            path = os.path.join(folder, f)
            text = extract_text(path)

            # Print short preview in the notebook
            print(f"\n File: {f}")
            print(text[:400])  # show only first 400 characters
            print("-" * 50)

            # Save full text into the output file
            out.write(f"\n\n===== FILE: {f} =====\n")
            out.write(text)
            out.write("\n" + "=" * 80 + "\n")

print(f"All extracted texts saved to: {output_file}")



 File: 4fda3c73-1760388107295-71d268b4-014f-4b31-8103-e4c5ebb0d6ab_15.png
Given a set of observed points, or input-output examples, the
distribution of the (unobserved) output of a new point as
function of its input data can be directly computed by looking
like the observed points and the covariances between those
points and the new, unobserved point.

 

Gaussian processes are popular surrogate models in Bayesian : E

optimisation used to do hyperparameter optimisation
--------------------------------------------------

 File: 1cdd3e45-1760388107295-71d268b4-014f-4b31-8103-e4c5ebb0d6ab_5.png
Conventional statistical analyses require the a priori selection of a model most suitable for the study
data set. In addition, only significant or theoretically relevant variables based on previous experience
are included for analysis. In contrast, machine learning is not built on a pre-structured model; rather,
the data shape the model by detecting underlying patterns. The more variables (input)

# Text Embedding Using OpenAI

In [11]:
import os
os.environ['OPENAI_API_KEY'] ="API--KEY"


### Loading the Saved extracted texts

In [12]:
text_path = "/content/extracted_texts.txt"

with open(text_path, "r", encoding="utf-8") as f:
    full_text = f.read()

print("Loaded text length:", len(full_text))
print(full_text[:500])  # preview first 500 characters


Loaded text length: 188988


===== FILE: 4fda3c73-1760388107295-71d268b4-014f-4b31-8103-e4c5ebb0d6ab_15.png =====
Given a set of observed points, or input-output examples, the
distribution of the (unobserved) output of a new point as
function of its input data can be directly computed by looking
like the observed points and the covariances between those
points and the new, unobserved point.

 

Gaussian processes are popular surrogate models in Bayesian : E

optimisation used to do hyperparameter optimisation. ‘An example


### Split text into chunks (for better embedding)

In [13]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # each chunk ~800 characters
    chunk_overlap=200,   # small overlap to preserve context
    length_function=len
)

texts = splitter.split_text(full_text)
print(f" Created {len(texts)} text chunks for embedding.")
print(texts[0][:400])  # show first chunk


 Created 335 text chunks for embedding.
===== FILE: 4fda3c73-1760388107295-71d268b4-014f-4b31-8103-e4c5ebb0d6ab_15.png =====
Given a set of observed points, or input-output examples, the
distribution of the (unobserved) output of a new point as
function of its input data can be directly computed by looking
like the observed points and the covariances between those
points and the new, unobserved point.

 

Gaussian processes are popular 


### Using OpenAI embeddings

In [14]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectors = embeddings.embed_documents(texts)

print(" Generated embeddings shape:", len(vectors), "vectors")
print("Example vector length:", len(vectors[0]))


 Generated embeddings shape: 335 vectors
Example vector length: 1536


### Saving the Embedding

In [15]:
import numpy as np

np.save("/content/text_chunks.npy", texts)
np.save("/content/text_vectors.npy", vectors)

print(" Saved text chunks and embeddings.")


 Saved text chunks and embeddings.


### **Build A simple Search**

> Add blockquote



In [16]:
from openai import OpenAI

# Initialize the client
client = OpenAI(api_key=os.getenv("API-KEY"))  # or paste key directly if needed


### Load your saved embeddings and text chunks

In [17]:
texts = np.load("/content/text_chunks.npy", allow_pickle=True)
vectors = np.load("/content/text_vectors.npy", allow_pickle=True)

print(f"Loaded {len(texts)} text chunks and {vectors.shape[0]} embeddings.")


Loaded 335 text chunks and 335 embeddings.


##### Create a csv

In [18]:
import pandas as pd
df = pd.DataFrame({
    "text_chunk": texts,
    "values": [str(list(v)) for v in vectors],
    "metadata": [{"source": "extracted_texts.txt", "chunk": i} for i in range(len(texts))]
})

df.to_csv("/content/text_embeddings.csv", index=True)
print(" Saved CSV with text, embeddings, and metadata.")


 Saved CSV with text, embeddings, and metadata.


##### Reload the clean dataframe

In [19]:
def prepare_DF(df):
    import json, ast
    try:
        df = df.drop('Unnamed: 0', axis=1)
    except:
        print('Unnamed Not Found')
    df['values'] = df['values'].apply(lambda x: np.array([float(i) for i in x.replace("[",'').replace("]",'').split(',')]))
    df['metadata'] = df['metadata'].apply(lambda x: ast.literal_eval(x))
    return df


In [20]:
df = pd.read_csv("/content/text_embeddings.csv")
df_clean = prepare_DF(df)
df_clean.head()

,text_chunk,values,metadata
0,===== FILE: 4fda3c73-1760388107295-71d268b4-01...,"[-0.03676250949501991, 3.181349893566221e-05, ...","{'source': 'extracted_texts.txt', 'chunk': 0}"
1,optimisation used to do hyperparameter optimis...,"[-0.03533438220620155, 0.014518708921968937, 0...","{'source': 'extracted_texts.txt', 'chunk': 1}"
2,"The theory of belief functions, also referred ...","[0.007331287954002619, -0.014536864124238491, ...","{'source': 'extracted_texts.txt', 'chunk': 2}"
3,to Bayesian approaches in order to incorporate...,"[0.013153054751455784, -0.023170996457338333, ...","{'source': 'extracted_texts.txt', 'chunk': 3}"
4,Rule-based models\n\nRule-based machine learni...,"[-0.01635592058300972, 0.037814125418663025, 0...","{'source': 'extracted_texts.txt', 'chunk': 4}"


### Create a searchable vector index (FAISS)
#####(Facebook AI Similarity Search) for efficient semantic retrieval.

In [21]:
!pip install faiss-cpu

import faiss
import numpy as np

# Create a FAISS index
dimension = vectors.shape[1]  # e.g., 384 for MiniLM, 1536 for OpenAI
index = faiss.IndexFlatL2(dimension)
index.add(vectors.astype('float32'))

print("FAISS index built with", index.ntotal, "vectors.")


FAISS index built with 335 vectors.


### Define a search helper to find relevant chunks

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

def search_similar_chunks(query_vector, top_k=3):
    # Use FAISS search
    distances, indices = index.search(np.array([query_vector]).astype('float32'), top_k)
    return [(int(i), float(d)) for i, d in zip(indices[0], distances[0])]


### Ask questions against the vector database

In [23]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
query = "Who is sister Terri"
query_vector = np.array(embeddings.embed_query(query))


In [24]:
query_vector

array([  0.0049623,  -0.0087697,   -0.071469, ...,   -0.019393, -0.00050265,   -0.019507])

##### Retrieve top-matching chunks

In [25]:
results = search_similar_chunks(query_vector, top_k=3)

print("\n Top relevant text chunks:")
for idx, score in results:
    print(f"\n Chunk #{idx} (distance={score:.4f}):\n{texts[idx][:500]}")



 Top relevant text chunks:

 Chunk #316 (distance=1.4116):
and introduce you to your future partner.



===== FILE: f94898df-1760378000404-38a681d7-57f1-40cb-acc3-0c0e2b09c28b_203.png =====
SURVIVING THE SINGLES SCENE - 183

‘My sister Terri, the political science professor, shared this
story, complete with a getaway technique. “I once had a
date with a guy who was very opinionated. He assumed,
though he never asked, that I felt just as he did about
politics, religion, et 

 Chunk #255 (distance=1.5559):
reveals his frustration in the telling of this tale: “A
‘friend’ fixed me up, a term I have learned to loathe, with
a psychologist from Manhattan who specializes in work-
ing with sexually abused women and children. I like
smart women, so I (foolishly) said yes. I met Sarah at a
wonderful restaurant in San Francisco. After we sat
down, her first question was ‘Have you had an AIDS
test?’ (I was immediately reminded of a woman friend
who suggested, at the time of my divorce, that I hav

### Send context + question to an LLM for final answer

In [26]:
from openai import OpenAI
import os, textwrap

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Combine the top relevant chunks as context
context = "\n\n".join([texts[i] for i, _ in results])

# Create the prompt
prompt = f"""
You are an expert medical document assistant.
Use the following extracted information to answer concisely and accurately.

Context:
{context}

Question: {query}
Answer:
"""

# Run the model
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[{"role": "user", "content": prompt}]
)

# Get the response text
answer = completion.choices[0].message.content.strip()

# Nicely wrap the text to new lines if too long
wrapped_answer = textwrap.fill(answer, width=90)

print("\nFinal Answer:\n")
print(wrapped_answer)



Final Answer:

Sister Terri is a political science professor.


### GRADIO UI

In [27]:
#Install dependencies
!pip -q install gradio faiss-cpu langchain langchain-openai tiktoken

# Imports and setup
import os
import numpy as np
import faiss
import gradio as gr
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

#  Set your OpenAI key (if not set already)
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your key or set it in Colab env

# Load precomputed embeddings and text chunks
texts = np.load("/content/text_chunks.npy", allow_pickle=True)
vectors = np.load("/content/text_vectors.npy", allow_pickle=True).astype("float32")
texts = texts.tolist() if isinstance(texts, np.ndarray) else texts
dim = vectors.shape[1]

# Build FAISS index
index = faiss.IndexFlatL2(dim)
index.add(vectors)
print(f" FAISS index built with {len(texts)} text chunks.")

# Initialize models
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# # Retrieval + QA function
def get_answer(query, history=[]):
    # Embed query and retrieve top chunks
    q_vec = np.array(embeddings.embed_query(query), dtype="float32").reshape(1, -1)
    D, I = index.search(q_vec, 3)
    context = "\n\n".join([texts[i] for i in I[0]])

    # Ask model
    prompt = f"""
    You are a helpful medical assistant. Use only the context below to answer the question.

    Context:
    {context}

    Question: {query}
    Answer:
    """
    try:
        response = llm.invoke([{"role": "user", "content": prompt}])
        answer = response.content.strip()
    except Exception as e:
        answer = f"Error: {e}"

    history.append((query, answer))
    return "", history


# Gradio Chatbot UI
with gr.Blocks(title="Fertility Clinic Chatbot") as demo:
    gr.Markdown("## Fertility Clinic Chatbot")
    gr.Markdown("Ask questions based on the embedded medical documents (preloaded).")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(placeholder="Ask a question about the documents...", label="Your Question")

    clear = gr.Button("Clear Chat")

    msg.submit(get_answer, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)



 FAISS index built with 335 text chunks.


/tmp/ipython-input-32094131.py:62: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)


In [28]:
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73c2326beef6f4185f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
